# Training

Pretrained models are in `models/`. This notebook can retrain every part; all flags are off by default.

| Flag | What is trained | Input | Time (CPU) |
|---|---|---|---|
| `TRAIN_MODEL_A` | the 5 Model A components, 12 months | features | ~4–5 h (mostly CatBoost) |
| `TRAIN_OOF` | Model A OOF: 3 splits × 5 components (train on 10 months, predict 2) → `artifacts/oof/model_a_oof.parquet` | features | a few hours per split |
| `TRAIN_CORRECTORS` | ADS-B, OPDI and LTFM second-stage correctors | OOF + `adsb_tracks`, `adsb_tracks_s30`, `adsb_tracks_s10`, OPDI | ~10 min |
| `TRAIN_MODEL_B` | Model B base (its ADS-B and residual correctors are trained at prediction time) | features | ~2 min |
| `TRAIN_MODEL_B_OOF` | Model B OOF (training base of the residual corrector) → `artifacts/oof/model_b_oof.parquet` | features + ADS-B | ~15 min |

Order matters: Model A and its OOF come before the correctors. With a GPU (`TAXI_GPU=1`), CatBoost and XGBoost run on the GPU.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
import config
import run
print("raw data folder:", config.RAW.name)

raw data folder: data


In [2]:
TRAIN_MODEL_A = False
TRAIN_OOF = False
TRAIN_CORRECTORS = False
TRAIN_MODEL_B = False
TRAIN_MODEL_B_OOF = False

In [3]:
if TRAIN_MODEL_A:
    run.train_a()
if TRAIN_OOF:
    run.train_a_oof()
if TRAIN_CORRECTORS:
    run.train_correctors()
if TRAIN_MODEL_B:
    run.train_b()
if TRAIN_MODEL_B_OOF:
    run.train_b_oof()
print("done")

done


## Blend weights
Weights selected by cross-split SLSQP on the provided OOF, next to the weights in `config.MODEL_A`.

In [4]:
run.weights()

[      1 s] chosen weights: xgboost 0.1949, catboost_symmetric 0.1484, lightgbm_lock 0.2204, lightgbm_huber 0.1289, catboost_depthwise 0.3073


[      1 s] config.MODEL_A: xgboost 0.1021, catboost_symmetric 0.2036, lightgbm_lock 0.2211, lightgbm_huber 0.1236, catboost_depthwise 0.3497


{'xgboost': 0.1949,
 'catboost_symmetric': 0.1484,
 'lightgbm_lock': 0.2204,
 'lightgbm_huber': 0.1289,
 'catboost_depthwise': 0.3073}

## Saved models

In [5]:
rows = []
for k in sorted(p for p in config.MODELS.rglob("*") if p.is_file()):
    rows.append((str(k.parent.relative_to(config.MODELS)), k.name, k.stat().st_size / 1e6))
m = pd.DataFrame(rows, columns=["folder", "file", "MB"])
m.groupby("folder").agg(files=("file", "size"), MB=("MB", "sum")).round(1)

,files,MB
folder,,
adsb,6,57.5
ltfm,4,6.8
model_a/catboost_depthwise,5,311.2
model_a/catboost_symmetric,4,299.7
model_a/lightgbm_huber,31,817.2
model_a/lightgbm_lock,41,844.8
model_a/xgboost,21,339.5
model_b,11,28.6
opdi,4,10.1
